In [ ]:
# ============================================================
#  EMA 9/21 + ADX CROSSOVER BACKTEST — BTC-USD 5M
#  Strategy:
#    BUY  when EMA9 > EMA21 AND ADX > 20 (trend confirmed)
#    SELL when EMA9 < EMA21 AND ADX > 20 (trend confirmed)
#    HOLD when ADX < 20 (no trend = skip signal)
#  Lot Cost : Rs 1000 per trade
#  Period   : 1 month BTC-USD 5M
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np

# ── CONFIG ────────────────────────────────────────────────────────────────
TICKER   = "BTC-USD"
PERIOD   = "1mo"
INTERVAL = "5m"
LOT      = 1000
ADX_MIN  = 25      # ADX must be above this to take any trade
MIN_HOLD = 3       # minimum candles to hold before reversing

# ── DATA (live or simulated) ──────────────────────────────────────────────
try:
    print("Fetching BTC-USD from Yahoo Finance...")
    df = yf.download(TICKER, period=PERIOD, interval=INTERVAL, progress=False)
    df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
    df.dropna(inplace=True)
    if len(df) < 100: raise ValueError("Too few rows")
    print(f"Loaded {len(df)} candles")
except:
    print("[Simulated BTC-USD 5M data]")
    np.random.seed(42); N=8640; BTC=62000
    ret = np.random.normal(0, 0.0015, N)
    ret[500:1200]  += 0.0008   # bull trend
    ret[2000:2800] -= 0.0012   # bear trend
    ret[3500:4500] += 0.0006   # recovery
    ret[5500:6200] -= 0.0008   # selloff
    ret[7000:8000] += 0.0005   # rally
    ret[1200:1800]  = np.random.normal(0, 0.0004, 600)   # choppy
    ret[4500:5500]  = np.random.normal(0, 0.0004, 1000)  # choppy
    close = BTC * np.cumprod(1 + ret)
    high  = close * (1 + np.random.uniform(0.001, 0.005, N))
    low   = close * (1 - np.random.uniform(0.001, 0.005, N))
    op    = np.roll(close, 1); op[0] = BTC
    idx   = pd.date_range("2025-05-01 09:00", periods=N, freq="5min")
    df    = pd.DataFrame({'Open':op,'High':high,'Low':low,
                          'Close':close,'Volume':np.ones(N)*1000}, index=idx)

# ── INDICATORS ────────────────────────────────────────────────────────────
h = df['High']; l = df['Low']; c = df['Close']

# EMA 9 and EMA 21
df['EMA9']  = c.ewm(span=9,  adjust=False).mean()
df['EMA21'] = c.ewm(span=21, adjust=False).mean()

# ATR (needed for ADX and position sizing)
tr  = pd.concat([h - l,
                 (h - c.shift(1)).abs(),
                 (l - c.shift(1)).abs()], axis=1).max(axis=1)
df['ATR'] = tr.ewm(span=14, adjust=False).mean()

# ADX (+DI, -DI, DX, ADX)
up_move   = h.diff()
down_move = l.shift(1) - l

plus_dm   = pd.Series(np.where((up_move > down_move) & (up_move > 0), up_move, 0.0), index=df.index)
minus_dm  = pd.Series(np.where((down_move > up_move) & (down_move > 0), down_move, 0.0), index=df.index)

atr14     = tr.rolling(14).mean().replace(0, np.nan)
plus_di   = 100 * plus_dm.rolling(14).mean()  / atr14
minus_di  = 100 * minus_dm.rolling(14).mean() / atr14
dx        = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)
df['ADX']      = dx.rolling(14).mean()
df['PLUS_DI']  = plus_di
df['MINUS_DI'] = minus_di

# EMA Gap (how far apart are the two EMAs — small gap = noise)
df['EMA_GAP'] = (df['EMA9'] - df['EMA21']).abs()

# Raw EMA crossover
fast = df['EMA9']; slow = df['EMA21']
df['CROSS'] = np.where(
    (fast > slow) & (fast.shift(1) <= slow.shift(1)), 'GOLDEN',
    np.where(
        (fast < slow) & (fast.shift(1) >= slow.shift(1)), 'DEATH',
        'NONE'))

df.dropna(inplace=True)

# ── STRATEGY LOGIC ────────────────────────────────────────────────────────
# Signal generation with ADX filter:
#   GOLDEN cross + ADX > 20 + +DI > -DI  → BUY
#   DEATH  cross + ADX > 20 + -DI > +DI  → SELL
#   ADX < 20 → SKIP (ranging market, whipsaw risk)

def run_backtest(df, adx_min=20, min_hold=3):
    trades  = []
    pos     = None          # None | 'BUY' | 'SELL'
    ep      = 0.0           # entry price
    ei      = 0             # entry index
    et      = None          # entry time
    skipped = 0             # count of skipped signals

    for i in range(1, len(df)):
        row   = df.iloc[i]
        price = float(row['Close'])
        cross = str(row['CROSS'])
        adx   = float(row['ADX'])
        pdi   = float(row['PLUS_DI'])
        mdi   = float(row['MINUS_DI'])
        atr   = float(row['ATR'])
        gap   = float(row['EMA_GAP'])
        t     = df.index[i]

        if cross == 'NONE':
            continue

        # ── ADX FILTER: skip if no trend ─────────────────────────────────
        if adx < adx_min:
            skipped += 1
            continue

        # ── MIN HOLD: don't reverse too quickly ───────────────────────────
        if pos is not None and (i - ei) < min_hold:
            skipped += 1
            continue

        # ── GOLDEN CROSS → BUY ───────────────────────────────────────────
        if cross == 'GOLDEN' and pdi > mdi:   # +DI confirms bullish

            # Close existing SELL position first
            if pos == 'SELL':
                pnl = ep - price
                ret = LOT + (LOT * pnl / ep)
                trades.append({
                    'trade_no'    : len(trades) + 1,
                    'action'      : 'SELL→CLOSE+BUY',
                    'entry_time'  : et,
                    'exit_time'   : t,
                    'entry_price' : round(ep, 2),
                    'exit_price'  : round(price, 2),
                    'hold_candles': i - ei,
                    'adx'         : round(adx, 2),
                    'plus_di'     : round(pdi, 2),
                    'minus_di'    : round(mdi, 2),
                    'atr'         : round(atr, 2),
                    'invested'    : LOT,
                    'returned'    : round(ret, 2),
                    'pnl'         : round(ret - LOT, 2),
                    'result'      : 'WIN' if ret > LOT else 'LOSS',
                })
                pos = None

            # Open BUY
            pos = 'BUY'; ep = price; ei = i; et = t

        # ── DEATH CROSS → SELL ───────────────────────────────────────────
        elif cross == 'DEATH' and mdi > pdi:  # -DI confirms bearish

            # Close existing BUY position first
            if pos == 'BUY':
                pnl = price - ep
                ret = LOT + (LOT * pnl / ep)
                trades.append({
                    'trade_no'    : len(trades) + 1,
                    'action'      : 'BUY→CLOSE+SELL',
                    'entry_time'  : et,
                    'exit_time'   : t,
                    'entry_price' : round(ep, 2),
                    'exit_price'  : round(price, 2),
                    'hold_candles': i - ei,
                    'adx'         : round(adx, 2),
                    'plus_di'     : round(pdi, 2),
                    'minus_di'    : round(mdi, 2),
                    'atr'         : round(atr, 2),
                    'invested'    : LOT,
                    'returned'    : round(ret, 2),
                    'pnl'         : round(ret - LOT, 2),
                    'result'      : 'WIN' if ret > LOT else 'LOSS',
                })
                pos = None

            # Open SELL
            pos = 'SELL'; ep = price; ei = i; et = t

    # ── CLOSE OPEN POSITION AT END OF DATA ───────────────────────────────
    if pos is not None:
        last  = float(df['Close'].iloc[-1])
        pnl   = (last - ep) if pos == 'BUY' else (ep - last)
        ret   = LOT + (LOT * pnl / ep)
        last_adx = float(df['ADX'].iloc[-1])
        last_pdi = float(df['PLUS_DI'].iloc[-1])
        last_mdi = float(df['MINUS_DI'].iloc[-1])
        trades.append({
            'trade_no'    : len(trades) + 1,
            'action'      : f'{pos}→OPEN_AT_END',
            'entry_time'  : et,
            'exit_time'   : df.index[-1],
            'entry_price' : round(ep, 2),
            'exit_price'  : round(last, 2),
            'hold_candles': len(df) - ei,
            'adx'         : round(last_adx, 2),
            'plus_di'     : round(last_pdi, 2),
            'minus_di'    : round(last_mdi, 2),
            'atr'         : round(float(df['ATR'].iloc[-1]), 2),
            'invested'    : LOT,
            'returned'    : round(ret, 2),
            'pnl'         : round(ret - LOT, 2),
            'result'      : 'WIN' if ret > LOT else 'LOSS',
        })

    return pd.DataFrame(trades), skipped

# ── RUN ───────────────────────────────────────────────────────────────────
tdf, skipped = run_backtest(df, adx_min=ADX_MIN, min_hold=MIN_HOLD)

# ── STATS ─────────────────────────────────────────────────────────────────
total   = len(tdf)
wins    = int((tdf['result'] == 'WIN').sum())
losses  = int((tdf['result'] == 'LOSS').sum())
put     = float(tdf['invested'].sum())
got     = float(tdf['returned'].sum())
pnl     = got - put
wr      = wins / total * 100 if total > 0 else 0
gp      = float(tdf[tdf['pnl'] > 0]['pnl'].sum()) if wins > 0 else 0.0
gl      = float(abs(tdf[tdf['pnl'] < 0]['pnl'].sum())) if losses > 0 else 0.001
pf      = gp / gl
aw      = float(tdf[tdf['result']=='WIN']['pnl'].mean())  if wins > 0   else 0.0
al      = float(tdf[tdf['result']=='LOSS']['pnl'].mean()) if losses > 0 else 0.0
best    = float(tdf['pnl'].max())
worst   = float(tdf['pnl'].min())
eq      = LOT + tdf['pnl'].cumsum()
pk      = eq.cummax()
mdd     = float(((eq - pk) / pk * 100).min())
avg_adx = float(tdf['adx'].mean())
avg_hc  = float(tdf['hold_candles'].mean())

total_crosses    = int((df['CROSS'] != 'NONE').sum())
adx_filter_pct  = skipped / total_crosses * 100 if total_crosses > 0 else 0

# ── PRINT REPORT ──────────────────────────────────────────────────────────
G="\033[92m"; R="\033[91m"; Y="\033[93m"; C="\033[96m"; RESET="\033[0m"; B="\033[1m"
pc = G if pnl >= 0 else R
wc = G if wr >= 50 else R

print(f"\n{B}{'='*64}{RESET}")
print(f"{B}   EMA 9 / EMA 21 + ADX  —  BTC-USD 5M  (1 Month){RESET}")
print(f"{B}{'='*64}{RESET}")

print(f"\n{B}   INDICATOR SETTINGS{RESET}")
print(f"   {'─'*56}")
print(f"   Fast EMA     : {C}EMA 9{RESET}")
print(f"   Slow EMA     : {C}EMA 21{RESET}")
print(f"   ADX Period   : {C}14{RESET}  (Wilder smoothing)")
print(f"   ADX Filter   : {C}ADX > {ADX_MIN}{RESET}  (skip signal if below)")
print(f"   DI Confirm   : {C}+DI > -DI for BUY  |  -DI > +DI for SELL{RESET}")
print(f"   Min Hold     : {C}{MIN_HOLD} candles = {MIN_HOLD*5} minutes{RESET}")
print(f"   Lot Cost     : {C}Rs {LOT}{RESET} per trade")

print(f"\n{B}   HOW ADX IS USED IN THIS STRATEGY{RESET}")
print(f"   {'─'*56}")
print(f"   {G}BUY  signal{RESET}  =  EMA9 > EMA21  AND  ADX > {ADX_MIN}  AND  +DI > -DI")
print(f"   {R}SELL signal{RESET}  =  EMA9 < EMA21  AND  ADX > {ADX_MIN}  AND  -DI > +DI")
print(f"   {Y}SKIP signal{RESET}  =  ADX < {ADX_MIN}  (ranging/choppy, no trend)")
print(f"")
print(f"   Total EMA crosses   : {total_crosses}")
print(f"   Skipped (ADX < {ADX_MIN})  : {skipped}  ({adx_filter_pct:.1f}% filtered out)")
print(f"   Trades taken        : {total}")
print(f"   Avg ADX at entry    : {avg_adx:.1f}")

print(f"\n{B}   TRADE SUMMARY{RESET}")
print(f"   {'─'*56}")
print(f"   Total Trades   : {B}{total}{RESET}")
print(f"   Win  Trades ✅ : {B}{G}{wins}{RESET}")
print(f"   Loss Trades ❌ : {B}{R}{losses}{RESET}")
print(f"   Win Rate       : {wc}{B}{wr:.1f}%{RESET}")

print(f"\n{B}   FINANCIAL SUMMARY{RESET}")
print(f"   {'─'*56}")
print(f"   Total PUT      : {Y}{B}Rs {put:>12,.2f}{RESET}  ({total} trades × Rs{LOT})")
print(f"   Total GOT      : {Y}{B}Rs {got:>12,.2f}{RESET}")
print(f"   Net P&L        : {pc}{B}Rs {pnl:>+12,.2f}{RESET}  ({pnl/put*100:+.2f}%)")
print(f"   Gross Profit   : {G}Rs {gp:>12,.2f}{RESET}")
print(f"   Gross Loss     : {R}Rs {gl:>12,.2f}{RESET}")
print(f"   Profit Factor  : {B}{pf:.2f}{RESET}  (>1.5 good | >2.0 excellent)")

print(f"\n{B}   TRADE QUALITY{RESET}")
print(f"   {'─'*56}")
print(f"   Avg Win        : {G}Rs {aw:>+10.2f}{RESET}")
print(f"   Avg Loss       : {R}Rs {al:>+10.2f}{RESET}")
print(f"   Win/Loss Ratio : {B}{abs(aw/al):.2f}x{RESET}  (avg win is N× avg loss)")
print(f"   Best Trade     : {G}Rs {best:>+10.2f}{RESET}")
print(f"   Worst Trade    : {R}Rs {worst:>+10.2f}{RESET}")
print(f"   Max Drawdown   : {R}{mdd:.2f}%{RESET}")
print(f"   Avg Hold       : {avg_hc:.1f} candles = {avg_hc*5:.0f} min")

print(f"\n{B}   FULL TRADE LOG (last 20 trades){RESET}")
print(f"   {'─'*90}")
print(f"   {'#':<5}{'Action':<22}{'Entry $':>10}{'Exit $':>10}{'ADX':>7}{'+DI':>7}{'-DI':>7}{'Bars':>6}{'P&L':>10}  Result")
print(f"   {'─'*90}")
for _, r in tdf.tail(20).iterrows():
    sym = "✅" if r['result']=='WIN' else "❌"
    cl  = G if r['result']=='WIN' else R
    di_winner = f"{G}+DI{RESET}" if r['plus_di'] > r['minus_di'] else f"{R}-DI{RESET}"
    print(f"   {int(r['trade_no']):<5}{r['action']:<22}"
          f"${r['entry_price']:>9,.0f}${r['exit_price']:>9,.0f}"
          f"{r['adx']:>7.1f}{r['plus_di']:>7.1f}{r['minus_di']:>7.1f}"
          f"{int(r['hold_candles']):>5}c "
          f"{cl}Rs{r['pnl']:>+9.2f}{RESET}  {sym}")

print(f"\n{B}   ADX FILTER IMPACT SUMMARY{RESET}")
print(f"   {'─'*56}")
print(f"   Without ADX filter  : ~{total_crosses} trades (all crosses)")
print(f"   With ADX > {ADX_MIN} filter : {total} trades taken  |  {skipped} skipped")
print(f"   Whipsaws avoided    : ~{skipped} low-quality entries filtered")
print(f"   Avg ADX at entries  : {avg_adx:.1f}  (all in confirmed trend)")

print(f"\n{'='*64}")
pc2 = G if pnl > 0 else R
verdict = "PROFITABLE ✅" if pnl > 0 else "LOSS ❌"
print(f"  {B}VERDICT: {pc2}{verdict}{RESET}")
print(f"  Net P&L  : {pc}{B}Rs {pnl:+,.2f}{RESET}  on Rs {put:,.0f} invested")
print(f"  Win Rate : {wr:.1f}%  |  Profit Factor : {pf:.2f}  |  Max DD : {mdd:.2f}%")
print(f"{'='*64}\n")

# Save trade log
tdf.to_csv("ema_adx_trades.csv", index=False)
print("  Full log saved to: ema_adx_trades.csv")
print("  Columns: trade_no, action, entry_time, exit_time,")
print("           entry_price, exit_price, hold_candles,")
print("           adx, plus_di, minus_di, atr, invested, returned, pnl, result")

Fetching BTC-USD from Yahoo Finance...


/tmp/ipykernel_7441/883405209.py:26: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(TICKER, period=PERIOD, interval=INTERVAL, progress=False)


Loaded 8928 candles

   EMA 9 / EMA 21 + ADX  —  BTC-USD 5M  (1 Month)

   INDICATOR SETTINGS
   ────────────────────────────────────────────────────────
   Fast EMA     : EMA 9
   Slow EMA     : EMA 21
   ADX Period   : 14  (Wilder smoothing)
   ADX Filter   : ADX > 25  (skip signal if below)
   DI Confirm   : +DI > -DI for BUY  |  -DI > +DI for SELL
   Min Hold     : 3 candles = 15 minutes
   Lot Cost     : Rs 1000 per trade

   HOW ADX IS USED IN THIS STRATEGY
   ────────────────────────────────────────────────────────
   BUY  signal  =  EMA9 > EMA21  AND  ADX > 25  AND  +DI > -DI
   SELL signal  =  EMA9 < EMA21  AND  ADX > 25  AND  -DI > +DI
   SKIP signal  =  ADX < 25  (ranging/choppy, no trend)

   Total EMA crosses   : 424
   Skipped (ADX < 25)  : 229  (54.0% filtered out)
   Trades taken        : 98
   Avg ADX at entry    : 36.0

   TRADE SUMMARY
   ────────────────────────────────────────────────────────
   Total Trades   : 98
   Win  Trades ✅ : 40
   Loss Trades ❌ : 58
   Win